# 🧪 المختبر 02 — الاكتشاف الدوائي بـ RDKit

## من SMILES إلى قاعدة ليبنسكي

**الفصل العاشر** — تطوير الأدوية

### 🎯 ما ستنجزه
1. تحليل SMILES بـ RDKit
2. حساب الأوصاف الجزيئية
3. تطبيق Rule of 5
4. اكتشاف أن نجوm 2026 تخترق القاعدة (bRo5)


In [ ]:
!pip install rdkit -q
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.figsize'] = (12, 6)
print(f"✅ RDKit {Chem.rdBase.rdkitVersion}")

## 📚 قاعدة ليبنسكي

| المعيار | الحد |
|---|---|
| MW | ≤ 500 |
| LogP | ≤ 5 |
| HBD | ≤ 5 |
| HBA | ≤ 10 |

In [ ]:
def analyze(name, smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {
        'name': name,
        'MW': round(Descriptors.MolWt(mol), 1),
        'LogP': round(Descriptors.MolLogP(mol), 2),
        'HBD': Lipinski.NumHDonors(mol),
        'HBA': Lipinski.NumHAcceptors(mol),
        'violations': sum([
            Descriptors.MolWt(mol) > 500,
            Descriptors.MolLogP(mol) > 5,
            Lipinski.NumHDonors(mol) > 5,
            Lipinski.NumHAcceptors(mol) > 10,
        ])
    }

In [ ]:
DRUGS = [
    ('Aspirin', 'CC(=O)Oc1ccccc1C(=O)O'),
    ('Paracetamol', 'CC(=O)Nc1ccc(O)cc1'),
    ('Ibuprofen', 'CC(C)Cc1ccc(cc1)C(C)C(=O)O'),
    ('Metformin', 'CN(C)C(=N)NC(=N)N'),
    ('Orforglipron (2026)', 'FC(F)(F)c1cc(cc(c1)C1(NC(=O)N(C2=NC(=O)N(C=C2)C)C)C)CC1'),
]

results = [analyze(n, s) for n, s in DRUGS]
results = [r for r in results if r]
df = pd.DataFrame(results)
print(df.to_string(index=False))

In [ ]:
colors = ['#22c55e' if v <= 1 else '#ef4444' for v in df['violations']]
plt.scatter(df['MW'], df['LogP'], c=colors, s=200, alpha=0.7, edgecolors='black')
for _, r in df.iterrows():
    plt.annotate(r['name'], (r['MW'], r['LogP']), fontsize=9)
plt.axvline(500, color='red', linestyle='--', alpha=0.5, label='MW ≤ 500')
plt.axhline(5, color='red', linestyle='--', alpha=0.5, label='LogP ≤ 5')
plt.xlabel('Molecular Weight (Da)')
plt.ylabel('LogP')
plt.title('Rule of 5: Traditional vs 2026 Stars')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 🎯 الدرس

Orforglipron (883 Da) يخترق قاعدة MW — لكنه **دواء فموي معتمد في 2026**.
**الاستثناء:** bRo5 (beyond Rule of 5) — فئات جديدة تحتاج مساحة كيميائية أكبر.